# LDA Preprocessing Pipeline

## spaCy over NLTK 
- Speed: spaCY is faster than nltk for large scale during tokenisation
- Better lemmatisation: no need to set up POS stagging pipeline then wordness lemmatizer, word2vec can auto infer POS
- using en_core_web_lg model from spaC bc its faster
- ref: https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf

In [15]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from spacy.language import Language
from spacy.schemas import ConfigSchemaNlp
ConfigSchemaNlp.model_rebuild()
nlp = spacy.load('en_core_web_lg')


# Tokenize & Lemmatize with spaCy

In [16]:
def preprocess_docs(texts: list[str], batch_size: int = 500) -> list[str]:
    """
    - Filters out: punctuation, whitespace, stop words, short tokens (<3 chars).
    - Returns one lemmatized string per document.
    """
    processed = []
 
    for doc in nlp.pipe(texts, disable=["parser", "ner"], batch_size=batch_size):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_space         # drop whitespace tokens
            and not token.is_stop          # drop stop words
            and len(token.lemma_) >= 3     # drop very short tokens
            and token.lemma_.isalpha()     # keep alphabetic tokens only
        ]
        processed.append(" ".join(tokens))
 
    return processed

# Vectorise with CountVectorizer

In [17]:
def build_count_matrix(processed_docs: list[str]):
    """
    Convert lemmatized documents into a sparse document-term count matrix.
    """
    vectorizer = CountVectorizer(
        min_df=5,                # token must appear in at least 5 documents -- remove rare words
        max_df=0.90,             # ignore tokens in >90% of documents -- remove too common words
        max_features=50_000,     # cap vocabulary size
        ngram_range=(1, 3),      # allow uni to tri-grams
    )
 
    dtm = vectorizer.fit_transform(processed_docs)
    return dtm, vectorizer

# Run Processing Pipeline

In [18]:
docs = pd.read_csv('../datasets/final/mda_shared_processed.csv')
docs.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentences,clean_text,n_sentences
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,49
1,1-800-PetMeds_10-K_2020-05-26,1-800-PetMeds,10-K,2020-05-26,2020,Q2,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,76
2,1-800-PetMeds_10-Q_2020-08-03,1-800-PetMeds,10-Q,2020-08-03,2020,Q3,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,45
3,1-800-PetMeds_10-Q_2020-11-03,1-800-PetMeds,10-Q,2020-11-03,2020,Q4,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,51
4,1-800-PetMeds_10-Q_2021-01-26,1-800-PetMeds,10-Q,2021-01-26,2021,Q1,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,48


In [ ]:
print("Step 1: Lemmatizing with spaCy...")
docs['processed_text'] = preprocess_docs(docs['clean_text'].tolist())

print("Step 2: Building document-term count matrix...")
dtm = build_count_matrix(docs['processed_text'])

print(f"Done. DTM shape: {dtm.shape}")

Step 1: Lemmatizing with spaCy...


In [ ]:
doc_index = docs[['doc_id', 'company_name', 'year', 'quarter']].reset_index(drop=True)